In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

print("🤖 INICIANDO PRUEBA DE MODELO: SUPPORT VECTOR REGRESSION (SVR)")
print("-" * 60)

# ==========================================
# 1. CARGA Y LIMPIEZA (El mismo pipeline de siempre)
# ==========================================
def limpiar_socio(path, pivot_col):
    df = pd.read_csv(path)
    if 'Poblacion' in path:
        df.columns = [str(col).split()[-1] if '01 ene' in str(col).lower() else col for col in df.columns]
    
    cols_num = [c for c in df.columns if c not in ['Territorio', 'Tipo de territorio', pivot_col]]
    for col in cols_num:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False).str.replace('-', '0', regex=False).replace('', '0').astype(float)
    
    melted = df.melt(id_vars=['Territorio', 'Tipo de territorio', pivot_col], var_name='Año', value_name='Val')
    melted['Año'] = pd.to_numeric(melted['Año'], errors='coerce')
    return melted.pivot_table(index=['Territorio', 'Año'], columns=pivot_col, values='Val', aggfunc='first').reset_index()

df_edad = limpiar_socio('Edad_Barrios.csv', 'Edad en grandes grupos')
df_pob = limpiar_socio('Poblacion-Inmigrante.csv', 'Nacionalidad (España/UE/Resto extranjero)')
df_crime = pd.read_csv('Criminalidad_Mensual_Estructurada.csv')

df_m = pd.merge(df_crime, df_edad, on=['Territorio', 'Año'], how='left')
df_m = pd.merge(df_m, df_pob, on=['Territorio', 'Año'], how='left')

df_m = df_m.rename(columns={'<16 años': 'Menos_16_anios', '≥65 años': 'Mas_65_anios', '16-64 años': 'Entre_16_64_anios'})
social_cols = ['Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

df_m = df_m.sort_values(['Territorio', 'Categoría_Delito', 'Año', 'Mes_Num'])
df_m[social_cols] = df_m.groupby(['Territorio', 'Categoría_Delito'])[social_cols].ffill()
df_m = df_m.dropna(subset=social_cols).reset_index(drop=True)

# ==========================================
# 2. FEATURE ENGINEERING (Igual que con LightGBM)
# ==========================================
def crear_variables_avanzadas(df):
    df = df.copy()
    df['Lag_1'] = df.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].shift(1)
    df['Lag_2'] = df.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].shift(2)
    df['Lag_3'] = df.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].shift(3)
    df['Lag_11'] = df.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].shift(11)
    df['Lag_12'] = df.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].shift(12)
    df['Media_3_meses'] = df.groupby(['Territorio', 'Categoría_Delito'])['Lag_1'].transform(lambda x: x.rolling(window=3).mean())
    df['Media_6_meses'] = df.groupby(['Territorio', 'Categoría_Delito'])['Lag_1'].transform(lambda x: x.rolling(window=6).mean())
    df['Media_12_meses'] = df.groupby(['Territorio', 'Categoría_Delito'])['Lag_1'].transform(lambda x: x.rolling(window=12).mean())
    df['Std_3_meses'] = df.groupby(['Territorio', 'Categoría_Delito'])['Lag_1'].transform(lambda x: x.rolling(window=3).std())
    df['Dif_Interanual'] = df['Lag_1'] - df['Lag_12']
    df['Poblacion_Total'] = df['España'] + df['Resto del mundo']
    return df.dropna().reset_index(drop=True)

df_model = crear_variables_avanzadas(df_m)

# ==========================================
# 3. PREPARACIÓN Y ESCALADO (Vital para SVM)
# ==========================================
print("⚙️ Preparando y escalando datos para el SVR...")
features = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_11', 'Lag_12', 
            'Media_3_meses', 'Media_6_meses', 'Media_12_meses', 'Std_3_meses', 'Dif_Interanual',
            'Mes_Num', 'Poblacion_Total',
            'Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

data_eval = df_model[(df_model['Territorio'] == 'Eixample') & (df_model['Categoría_Delito'] == 'Hurto')].copy()
X_eval = data_eval[features].values
y_eval = data_eval['Cantidad'].values

# Separación 80/20
split = int(len(X_eval) * 0.8)
X_train, X_test = X_eval[:split], X_eval[split:]
y_train, y_test = y_eval[:split], y_eval[split:]

# 🚨 El paso mágico para SVM: Estandarizar los datos (Media 0, Varianza 1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 4. ENTRENAMIENTO SVM (SVR)
# ==========================================
print("🧠 Entrenando la Máquina de Vectores de Soporte...")
# Usamos un kernel Radial (RBF) para intentar capturar formas no lineales
svm_model = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
svm_model.fit(X_train_scaled, y_train)

# Predicción
y_pred_svm = svm_model.predict(X_test_scaled)
y_pred_svm = [max(0, p) for p in y_pred_svm] # Evitar crímenes negativos

# ==========================================
# 5. EVALUACIÓN Y GRÁFICA
# ==========================================
mape_svm = mean_absolute_percentage_error(y_test, y_pred_svm) * 100
print(f"🎯 PRECISIÓN VALIDADA SVM: MAPE de {mape_svm:.2f}%")

plt.figure(figsize=(11, 5))
plt.plot(y_test, label='Realidad Histórica', color='navy', marker='o', linewidth=2)
plt.plot(y_pred_svm, label=f'SVM Prediction (Error: {mape_svm:.2f}%)', color='crimson', linestyle='--', marker='x', linewidth=2)
plt.title("Validación de Precisión SVM: Hurtos en L'Eixample", fontsize=15, fontweight='bold')
plt.ylabel("Incidentes")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("-" * 60)
print("COMPETICIÓN: Revisa el MAPE de este SVM y compáralo con el 5.86% del LightGBM.")

In [ ]:
from sklearn.model_selection import GridSearchCV

# ==========================================
# 3. PREPARACIÓN Y ESCALADO
# ==========================================
print("⚙️ Preparando y escalando datos para el SVR...")
features = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_11', 'Lag_12', 
            'Media_3_meses', 'Media_6_meses', 'Media_12_meses', 'Std_3_meses', 'Dif_Interanual',
            'Mes_Num', 'Poblacion_Total',
            'Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

data_eval = df_model[(df_model['Territorio'] == 'Eixample') & (df_model['Categoría_Delito'] == 'Hurto')].copy()
X_eval = data_eval[features].values
y_eval = data_eval['Cantidad'].values

split = int(len(X_eval) * 0.8)
X_train, X_test = X_eval[:split], X_eval[split:]
y_train, y_test = y_eval[:split], y_eval[split:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 4. GRID SEARCH: BÚSQUEDA DEL SVM PERFECTO
# ==========================================
print("🧠 Iniciando Grid Search: Probando combinaciones de hiperparámetros...")

# Definimos la "cuadrícula" de opciones que la IA debe probar
param_grid = {
    'C': [10, 50, 100, 500],             # Fuerza de la penalización por equivocarse
    'gamma': ['scale', 'auto', 0.1, 0.01], # Flexibilidad de la curva
    'epsilon': [0.01, 0.1, 0.5, 1.0]       # Ancho del tubo de margen de error
}

# Lanzamos la búsqueda (n_jobs=-1 usa todos los núcleos de tu ordenador para ir más rápido)
svm_grid = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=5, scoring='neg_mean_absolute_percentage_error', n_jobs=-1)
svm_grid.fit(X_train_scaled, y_train)

# Rescatamos al campeón de la búsqueda
best_svm = svm_grid.best_estimator_
print(f"✅ ¡Mejor SVM encontrado! Parámetros: {svm_grid.best_params_}")

# Predicción con el modelo optimizado
y_pred_svm = best_svm.predict(X_test_scaled)
y_pred_svm = [max(0, p) for p in y_pred_svm]

# ==========================================
# 5. EVALUACIÓN Y GRÁFICA FINAL
# ==========================================
mape_svm = mean_absolute_percentage_error(y_test, y_pred_svm) * 100
print(f"🎯 PRECISIÓN OPTIMIZADA SVM: MAPE de {mape_svm:.2f}%")

plt.figure(figsize=(11, 5))
plt.plot(y_test, label='Realidad Histórica', color='navy', marker='o', linewidth=2)
plt.plot(y_pred_svm, label=f'SVM Optimizado (Error: {mape_svm:.2f}%)', color='crimson', linestyle='--', marker='x', linewidth=2)
plt.title("Validación SVM Optimizado con Grid Search: L'Eixample", fontsize=15, fontweight='bold')
plt.ylabel("Incidentes")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()